# Modelling - LH/RH Classification (Local CPU/GPU)

This notebook trains multiple machine-learning models on wide features, compares CV/LOSO/hold-out metrics, and saves artifacts under results_local_cpu_gpu/modelling.
Default mode is smoke for fast local validation.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, cohen_kappa_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve
)

HAS_XGB = False
HAS_LGB = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except Exception:
    HAS_LGB = False

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# 1) CONFIG
BASE_DIR = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook")
RESULTS_DIR = BASE_DIR / "results_local_cpu_gpu" / "modelling"
MODELS_DIR = BASE_DIR / "results_local_cpu_gpu" / "saved_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PRIMARY = BASE_DIR / "results_local_cpu_gpu" / "preprocessing" / "features_lh_rh.csv"
INPUT_FALLBACK = BASE_DIR / "features_lh_rh.csv"
INPUT_CSV = INPUT_PRIMARY if INPUT_PRIMARY.exists() else INPUT_FALLBACK

RUN_MODE = "smoke"  # change to "full" for full run
RANDOM_STATE = 42
N_FOLDS = 3 if RUN_MODE == "smoke" else 5
N_SELECT_K = 30 if RUN_MODE == "smoke" else 50
SMOKE_MAX_PER_CLASS = 1000
SMOKE_MAX_SUBJECTS_LOSO = 15

def savefig(name: str):
    plt.tight_layout()
    path = RESULTS_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

print(f"Input CSV: {INPUT_CSV}")
if not INPUT_CSV.exists():
    raise FileNotFoundError("features_lh_rh.csv not found. Run preprocessing notebook first.")

# 2) LOAD DATA
df = pd.read_csv(INPUT_CSV)
if RUN_MODE == "smoke":
    sampled_parts = []
    for _, part in df.groupby("label"):
        take_n = min(len(part), SMOKE_MAX_PER_CLASS)
        sampled_parts.append(part.sample(take_n, random_state=RANDOM_STATE))
    df = pd.concat(sampled_parts, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

META_COLS = {"subject_id", "scenario_id", "scenario", "filename", "task", "label", "label_name"}
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values.astype(float)
y = df["label"].astype(int).values
groups = df["subject_id"].astype(str).values

print(f"Data shape: X={X.shape}, y={y.shape}")
print("Class distribution:")
print(df["label_name"].value_counts())

def make_pipeline(clf, n_features=N_SELECT_K):
    k = max(1, min(n_features, X.shape[1]))
    return Pipeline([
        ("scaler", RobustScaler()),
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("clf", clf),
    ])

# 3) DEFINE MODELS
MODELS = {
    "Logistic Regression": LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=120 if RUN_MODE == "smoke" else 200,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=RANDOM_STATE),
}

if HAS_XGB:
    MODELS["XGBoost"] = XGBClassifier(
        n_estimators=120 if RUN_MODE == "smoke" else 200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=1,
    )
if HAS_LGB:
    MODELS["LightGBM"] = LGBMClassifier(
        n_estimators=120 if RUN_MODE == "smoke" else 200,
        max_depth=5,
        learning_rate=0.05,
        random_state=RANDOM_STATE,
    )

print("Models:", list(MODELS.keys()))

# 4) STRATIFIED K-FOLD CV
min_class_count = int(pd.Series(y).value_counts().min())
n_splits = max(2, min(N_FOLDS, min_class_count))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
SCORING = ["accuracy", "balanced_accuracy", "f1", "roc_auc"]

cv_records = []
for model_name, clf in MODELS.items():
    pipe = make_pipeline(clf)
    scores = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=1, return_train_score=False)
    rec = {
        "Model": model_name,
        "Accuracy_mean": float(np.mean(scores["test_accuracy"])),
        "Accuracy_std": float(np.std(scores["test_accuracy"])),
        "BAC_mean": float(np.mean(scores["test_balanced_accuracy"])),
        "BAC_std": float(np.std(scores["test_balanced_accuracy"])),
        "F1_mean": float(np.mean(scores["test_f1"])),
        "F1_std": float(np.std(scores["test_f1"])),
        "AUC_mean": float(np.mean(scores["test_roc_auc"])),
        "AUC_std": float(np.std(scores["test_roc_auc"])),
    }
    cv_records.append(rec)

cv_df = pd.DataFrame(cv_records).sort_values("BAC_mean", ascending=False).reset_index(drop=True)

cv_table = cv_df.copy()
cv_table["Accuracy"] = cv_table.apply(lambda r: f"{r['Accuracy_mean']:.4f} ± {r['Accuracy_std']:.4f}", axis=1)
cv_table["Bal. Accuracy"] = cv_table.apply(lambda r: f"{r['BAC_mean']:.4f} ± {r['BAC_std']:.4f}", axis=1)
cv_table["F1"] = cv_table.apply(lambda r: f"{r['F1_mean']:.4f} ± {r['F1_std']:.4f}", axis=1)
cv_table["ROC-AUC"] = cv_table.apply(lambda r: f"{r['AUC_mean']:.4f} ± {r['AUC_std']:.4f}", axis=1)
cv_out = cv_table[["Model", "Accuracy", "Bal. Accuracy", "F1", "ROC-AUC"]]
display(cv_out)
cv_df.to_csv(RESULTS_DIR / "cv_results.csv", index=False)

# 5) MODEL COMPARISON CHART
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_specs = [
    ("Accuracy_mean", "Accuracy_std", "Accuracy"),
    ("BAC_mean", "BAC_std", "Balanced Accuracy"),
    ("F1_mean", "F1_std", "F1"),
    ("AUC_mean", "AUC_std", "ROC-AUC"),
]
for ax, (m_col, s_col, title) in zip(axes.ravel(), metric_specs):
    bars = ax.bar(cv_df["Model"], cv_df[m_col], yerr=cv_df[s_col], capsize=4, alpha=0.85)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    for b, val in zip(bars, cv_df[m_col]):
        ax.text(b.get_x() + b.get_width() / 2, val + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
savefig("model_comparison.png")

best_model_name = cv_df.iloc[0]["Model"]
print(f"Best model by CV BAC: {best_model_name}")
best_model = make_pipeline(MODELS[best_model_name])

# 6) LOSO EVALUATION
logo = LeaveOneGroupOut()
loso_rows = []
used_subjects = 0
for train_idx, test_idx in logo.split(X, y, groups=groups):
    subj = str(groups[test_idx][0])
    y_test_subj = y[test_idx]
    if len(np.unique(y_test_subj)) < 2:
        continue

    best_model.fit(X[train_idx], y[train_idx])
    pred = best_model.predict(X[test_idx])
    proba = best_model.predict_proba(X[test_idx])[:, 1]

    row = {
        "subject_id": subj,
        "Accuracy": accuracy_score(y_test_subj, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_test_subj, pred),
        "F1": f1_score(y_test_subj, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test_subj, proba),
    }
    loso_rows.append(row)

    used_subjects += 1
    if RUN_MODE == "smoke" and used_subjects >= SMOKE_MAX_SUBJECTS_LOSO:
        break

loso_df = pd.DataFrame(loso_rows).sort_values("subject_id") if loso_rows else pd.DataFrame(columns=["subject_id", "Accuracy", "Balanced_Accuracy", "F1", "ROC_AUC"])
loso_df.to_csv(RESULTS_DIR / "loso_results.csv", index=False)

if not loso_df.empty:
    print("LOSO mean ± std:")
    for col in ["Accuracy", "Balanced_Accuracy", "F1", "ROC_AUC"]:
        print(f"{col}: {loso_df[col].mean():.4f} ± {loso_df[col].std():.4f}")

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    loso_sorted = loso_df.sort_values("subject_id")
    axes[0].bar(loso_sorted["subject_id"], loso_sorted["Accuracy"], color="tab:blue")
    axes[0].set_title("LOSO Accuracy per Subject")
    axes[0].set_ylim(0, 1.05)

    axes[1].bar(loso_sorted["subject_id"], loso_sorted["Balanced_Accuracy"], color="tab:green")
    axes[1].set_title("LOSO Balanced Accuracy per Subject")
    axes[1].set_ylim(0, 1.05)

    axes[2].bar(loso_sorted["subject_id"], loso_sorted["ROC_AUC"], color="tab:orange")
    axes[2].set_title("LOSO ROC-AUC per Subject")
    axes[2].set_ylim(0, 1.05)
    axes[2].tick_params(axis="x", rotation=90)
    savefig("loso_per_subject.png")

# 7) HOLD-OUT EVALUATION (all models)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

holdout_rows = []
roc_data = []
fitted_models = {}
for model_name, clf in MODELS.items():
    pipe = make_pipeline(clf)
    pipe.fit(X_train, y_train)
    fitted_models[model_name] = pipe

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    holdout_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced_Acc": balanced_accuracy_score(y_test, pred),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "Cohen_Kappa": cohen_kappa_score(y_test, pred),
    })

    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_data.append((model_name, fpr, tpr, roc_auc_score(y_test, proba)))

# Best model report on hold-out
best_hold_pipe = fitted_models[best_model_name]
best_pred = best_hold_pipe.predict(X_test)
best_proba = best_hold_pipe.predict_proba(X_test)[:, 1]
print("\nClassification report for best CV model on hold-out:")
print(classification_report(y_test, best_pred, target_names=["LH", "RH"], zero_division=0))
print(f"Cohen's Kappa: {cohen_kappa_score(y_test, best_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, best_proba):.4f}")

# 8) CONFUSION MATRIX
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, best_pred), display_labels=["LH", "RH"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion Matrix (Counts)")

cm_norm = confusion_matrix(y_test, best_pred, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=["LH", "RH"]).plot(ax=axes[1], colorbar=False)
axes[1].set_title("Confusion Matrix (Normalized)")
savefig("confusion_matrix.png")

# 9) ROC CURVES - ALL MODELS
plt.figure(figsize=(8, 6))
for model_name, fpr, tpr, auc_val in roc_data:
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Hold-out")
plt.legend(loc="lower right")
savefig("roc_curves.png")

# 10) FEATURE IMPORTANCE (best model)
best_full_pipe = make_pipeline(MODELS[best_model_name])
best_full_pipe.fit(X, y)
selector = best_full_pipe.named_steps["select"]
selected_mask = selector.get_support()
selected_features = np.array(feature_cols)[selected_mask]
clf = best_full_pipe.named_steps["clf"]

if hasattr(clf, "feature_importances_"):
    raw_imp = np.asarray(clf.feature_importances_)
elif hasattr(clf, "coef_"):
    raw_imp = np.abs(np.asarray(clf.coef_)).ravel()
else:
    # Fallback to selector scores when classifier has no direct importance API
    raw_imp = np.abs(np.asarray(selector.scores_)[selected_mask])

raw_imp = np.nan_to_num(raw_imp, nan=0.0, posinf=0.0, neginf=0.0)
if raw_imp.sum() > 0:
    norm_imp = raw_imp / raw_imp.sum()
else:
    norm_imp = raw_imp

importance_df = pd.DataFrame({
    "feature": selected_features,
    "importance": norm_imp,
}).sort_values("importance", ascending=False)
importance_df.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)

top30 = importance_df.head(30).sort_values("importance", ascending=True)
plt.figure(figsize=(10, 8))
plt.barh(top30["feature"], top30["importance"], color="tab:purple")
plt.xlabel("Normalized importance")
plt.title(f"Top 30 Feature Importance - {best_model_name}")
savefig("feature_importance.png")

# 11) IMPORTANCE HEATMAP (Channel x Subband)
heat_rows = []
for _, row in importance_df.iterrows():
    parts = str(row["feature"]).split("_", 2)
    if len(parts) < 3:
        continue
    ch, sb = parts[0], parts[1]
    heat_rows.append({"channel": ch, "subband": sb, "importance": row["importance"]})

heat_df = pd.DataFrame(heat_rows)
if not heat_df.empty:
    heat_pv = heat_df.groupby(["channel", "subband"], as_index=False)["importance"].sum().pivot(index="channel", columns="subband", values="importance")
    plt.figure(figsize=(8, 6))
    sns.heatmap(heat_pv, annot=True, cmap="YlOrRd", fmt=".3f")
    plt.title("Aggregated Importance Heatmap (Channel x Subband)")
    savefig("importance_heatmap.png")

# 12) FINAL METRICS TABLE
holdout_df = pd.DataFrame(holdout_rows)
cv_lookup = cv_df[["Model", "BAC_mean", "AUC_mean"]].rename(columns={"BAC_mean": "CV_BAC_mean", "AUC_mean": "CV_AUC_mean"})
final_df = holdout_df.merge(cv_lookup, on="Model", how="left")
final_df = final_df.sort_values("Balanced_Acc", ascending=False).reset_index(drop=True)
final_df.to_csv(RESULTS_DIR / "final_metrics.csv", index=False)
display(final_df)

heat_cols = ["Accuracy", "Balanced_Acc", "F1", "ROC_AUC", "Cohen_Kappa", "CV_BAC_mean", "CV_AUC_mean"]
metrics_heat = final_df.set_index("Model")[heat_cols]
plt.figure(figsize=(10, 5))
sns.heatmap(metrics_heat, annot=True, cmap="RdYlGn", vmin=0.3, vmax=1.0, fmt=".3f")
plt.title("Final Metrics Heatmap")
savefig("metrics_heatmap.png")

# 13) SAVE BEST MODEL + FEATURE LISTS
model_safe_name = best_model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
model_path = MODELS_DIR / f"{model_safe_name}_best.pkl"
joblib.dump(best_full_pipe, model_path)

feature_cols_path = MODELS_DIR / "feature_cols.txt"
with open(feature_cols_path, "w", encoding="utf-8") as f:
    for col in feature_cols:
        f.write(col + "\n")

selected_feature_path = MODELS_DIR / "selected_features_best_model.txt"
with open(selected_feature_path, "w", encoding="utf-8") as f:
    for col in selected_features:
        f.write(str(col) + "\n")

print("Saved result files:")
for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")
print("Saved model files:")
for p in sorted(MODELS_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")

Input CSV: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook/results_local_cpu_gpu/preprocessing/features_lh_rh.csv
Data shape: X=(300, 27), y=(300,)
Class distribution:
label_name
RH    150
LH    150
Name: count, dtype: int64
Models: ['Logistic Regression', 'Random Forest', 'SVM (RBF)', 'XGBoost', 'LightGBM']
[LightGBM] [Info] Number of positive: 100, number of negative: 100
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000694 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1092
[LightGBM] [Info] Number of data points in the train set: 200, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

,Model,Accuracy,Bal. Accuracy,F1,ROC-AUC
0,Logistic Regression,0.4467 ± 0.0249,0.4467 ± 0.0249,0.4537 ± 0.0238,0.4171 ± 0.0229
1,XGBoost,0.4333 ± 0.0368,0.4333 ± 0.0368,0.4199 ± 0.0566,0.3927 ± 0.0319
2,SVM (RBF),0.4300 ± 0.0566,0.4300 ± 0.0566,0.4423 ± 0.0613,0.3867 ± 0.0066
3,LightGBM,0.4233 ± 0.0170,0.4233 ± 0.0170,0.4203 ± 0.0354,0.4119 ± 0.0166
4,Random Forest,0.4200 ± 0.0141,0.4200 ± 0.0141,0.4199 ± 0.0111,0.3649 ± 0.0219


Best model by CV BAC: Logistic Regression
LOSO mean ± std:
Accuracy: 0.4333 ± 0.3716
Balanced_Accuracy: 0.4333 ± 0.3716
F1: 0.3333 ± 0.4364
ROC_AUC: 0.5333 ± 0.5164
[LightGBM] [Info] Number of positive: 120, number of negative: 120
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000436 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1233
[LightGBM] [Info] Number of data points in the train set: 240, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

,Model,Accuracy,Balanced_Acc,F1,ROC_AUC,Cohen_Kappa,CV_BAC_mean,CV_AUC_mean
0,SVM (RBF),0.416667,0.416667,0.426230,0.620000,-0.166667,0.430000,0.386667
1,LightGBM,0.400000,0.400000,0.379310,0.317778,-0.200000,0.423333,0.411867
2,Logistic Regression,0.366667,0.366667,0.406250,0.400000,-0.266667,0.446667,0.417067
3,XGBoost,0.316667,0.316667,0.305085,0.293333,-0.366667,0.433333,0.392667
4,Random Forest,0.283333,0.283333,0.271186,0.263333,-0.433333,0.420000,0.364933


Saved result files:
- confusion_matrix.png (41.5 KB)
- cv_results.csv (0.9 KB)
- feature_importance.csv (1.2 KB)
- feature_importance.png (231.4 KB)
- final_metrics.csv (0.7 KB)
- importance_heatmap.png (54.8 KB)
- loso_per_subject.png (94.1 KB)
- loso_results.csv (0.4 KB)
- metrics_heatmap.png (131.4 KB)
- model_comparison.png (167.4 KB)
- roc_curves.png (114.8 KB)
Saved model files:
- feature_cols.txt (0.6 KB)
- logistic_regression_best.pkl (2.5 KB)
- selected_features_best_model.txt (0.6 KB)
- tabular_eegnet_best.pt (9.3 KB)
- tabular_eegnet_feature_cols.txt (0.6 KB)
- tabular_eegnet_scaler.joblib (0.8 KB)
